# 3.7 Lab: FlashAttention in Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.7_flash_attention_practice/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.7_flash_attention_practice/lab.ipynb)

This lab benchmarks standard attention vs FlashAttention using PyTorch's SDPA dispatcher and HuggingFace transformers. You will measure latency and memory across sequence lengths.

In [ ]:
# Setup: install dependencies (flash-attn requires CUDA, ~2-3 min on Colab)
!pip install -q torch transformers accelerate flash-attn --no-build-isolation 2>/dev/null
!pip install -q matplotlib numpy

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt

# Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Experiment 1: SDPA Backend Comparison

PyTorch's `scaled_dot_product_attention` dispatches to different backends. We benchmark each explicitly across sequence lengths.

In [ ]:
# Parameters (change these and re-run to explore)
BATCH_SIZE = 4
NUM_HEADS = 32
HEAD_DIM = 128
SEQ_LENGTHS = [128, 256, 512, 1024, 2048, 4096, 8192]
WARMUP_ITERS = 10
BENCH_ITERS = 50
DTYPE = torch.float16

def benchmark_sdpa(seq_len, enable_flash, enable_math, enable_mem_efficient):
    """Benchmark SDPA with a specific backend configuration."""
    # Create Q, K, V in the shape SDPA expects: (B, H, N, D)
    q = torch.randn(BATCH_SIZE, NUM_HEADS, seq_len, HEAD_DIM, device=device, dtype=DTYPE)
    k = torch.randn(BATCH_SIZE, NUM_HEADS, seq_len, HEAD_DIM, device=device, dtype=DTYPE)
    v = torch.randn(BATCH_SIZE, NUM_HEADS, seq_len, HEAD_DIM, device=device, dtype=DTYPE)

    ctx = torch.backends.cuda.sdp_kernel(
        enable_flash=enable_flash,
        enable_math=enable_math,
        enable_mem_efficient=enable_mem_efficient
    )

    # Warmup to stabilize GPU clocks
    with ctx:
        for _ in range(WARMUP_ITERS):
            _ = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
    torch.cuda.synchronize()

    # Timed iterations
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with ctx:
        for _ in range(BENCH_ITERS):
            _ = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / BENCH_ITERS * 1000

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return elapsed_ms, peak_mb

print("Benchmarking SDPA backends...")

In [ ]:
# Run benchmarks for each backend
results = {'flash': [], 'math': [], 'efficient': []}

configs = {
    'flash': (True, False, False),
    'math': (False, True, False),
    'efficient': (False, False, True),
}

for seq_len in SEQ_LENGTHS:
    print(f"  seq_len={seq_len}...", end=" ")
    for name, (ef, em, eme) in configs.items():
        try:
            t, m = benchmark_sdpa(seq_len, ef, em, eme)
            results[name].append((seq_len, t, m))
            print(f"{name}={t:.2f}ms", end=" ")
        except RuntimeError:
            results[name].append((seq_len, None, None))
            print(f"{name}=N/A", end=" ")
    print()

print("Done.")

## Visualize: Latency vs Sequence Length

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'flash': '#2563eb', 'math': '#dc2626', 'efficient': '#16a34a'}
markers = {'flash': 'o', 'math': 's', 'efficient': '^'}
labels = {'flash': 'FlashAttention', 'math': 'Math (standard)', 'efficient': 'Memory-efficient'}

# Latency plot (log-log scale)
for name in ['flash', 'math', 'efficient']:
    valid = [(s, t) for s, t, m in results[name] if t is not None]
    if valid:
        seqs, times = zip(*valid)
        ax1.plot(seqs, times, f'{markers[name]}-', color=colors[name],
                 linewidth=2, label=labels[name])

ax1.set_xscale('log', base=2)
ax1.set_yscale('log')
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Attention Latency by Backend')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Speedup plot (flash vs math)
flash_valid = [(s, t) for s, t, m in results['flash'] if t is not None]
math_valid = [(s, t) for s, t, m in results['math'] if t is not None]
if flash_valid and math_valid:
    # Match by sequence length
    flash_dict = dict(flash_valid)
    math_dict = dict(math_valid)
    common = sorted(set(flash_dict.keys()) & set(math_dict.keys()))
    speedups = [math_dict[s] / flash_dict[s] for s in common]

    ax2.bar(range(len(common)), speedups, color='#2563eb', alpha=0.7)
    ax2.set_xticks(range(len(common)))
    ax2.set_xticklabels([str(s) for s in common])
    ax2.set_xlabel('Sequence Length')
    ax2.set_ylabel('Speedup (standard / flash)')
    ax2.set_title('FlashAttention Speedup over Standard')
    ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('flash_attention_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: flash_attention_benchmark.png")

## Experiment 2: HuggingFace Model with Flash vs Eager Attention

Load a real model with both attention implementations and measure prefill latency.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Parameters
MODEL_NAME = "microsoft/phi-2"  # Small enough for T4, supports flash_attention_2
PROMPT_LENGTHS = [128, 512, 1024, 2048]
NUM_RUNS = 5

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Loaded tokenizer for {MODEL_NAME}")

In [ ]:
def measure_prefill(model, input_ids, num_runs=5):
    """Measure average prefill latency in ms."""
    # Warmup
    with torch.no_grad():
        for _ in range(2):
            _ = model(input_ids)
    torch.cuda.synchronize()

    # Timed runs
    times = []
    for _ in range(num_runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(input_ids)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return np.mean(times), np.std(times)

# Load both model variants
print("Loading eager model...")
model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="eager", trust_remote_code=True
)
print("Loading flash model...")
model_flash = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
    attn_implementation="flash_attention_2", trust_remote_code=True
)
print("Both models loaded.")

In [ ]:
# Benchmark prefill for both implementations
results_eager = []
results_flash_model = []

for length in PROMPT_LENGTHS:
    input_ids = torch.randint(100, 30000, (1, length), device=device)
    print(f"  Prompt length={length}...", end=" ")

    mean_e, std_e = measure_prefill(model_eager, input_ids, NUM_RUNS)
    results_eager.append((length, mean_e, std_e))
    print(f"eager={mean_e:.1f}ms", end=" ")

    mean_f, std_f = measure_prefill(model_flash, input_ids, NUM_RUNS)
    results_flash_model.append((length, mean_f, std_f))
    print(f"flash={mean_f:.1f}ms  speedup={mean_e/mean_f:.2f}x")

# Free VRAM
del model_eager, model_flash
torch.cuda.empty_cache()

## Visualize: Model-Level Prefill Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

lengths = [r[0] for r in results_eager]
eager_times = [r[1] for r in results_eager]
eager_stds = [r[2] for r in results_eager]
flash_times = [r[1] for r in results_flash_model]
flash_stds = [r[2] for r in results_flash_model]

x = np.arange(len(lengths))
width = 0.35

# Bar chart: eager vs flash
ax.bar(x - width/2, eager_times, width, yerr=eager_stds,
       label='Eager (standard)', color='#dc2626', alpha=0.7, capsize=3)
ax.bar(x + width/2, flash_times, width, yerr=flash_stds,
       label='FlashAttention-2', color='#2563eb', alpha=0.7, capsize=3)

ax.set_xlabel('Prompt Length (tokens)')
ax.set_ylabel('Prefill Latency (ms)')
ax.set_title(f'{MODEL_NAME}: Eager vs FlashAttention-2 Prefill')
ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in lengths])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Annotate speedup above flash bars
for i in range(len(lengths)):
    speedup = eager_times[i] / flash_times[i]
    ax.annotate(f'{speedup:.1f}x', xy=(x[i] + width/2, flash_times[i]),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9, color='#2563eb')

plt.tight_layout()
plt.savefig('model_flash_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_flash_comparison.png")

## Key Takeaways

1. FlashAttention speedup grows with sequence length (negligible at N<256, essential at N>32K)
2. PyTorch SDPA automatically selects FlashAttention on compatible hardware and dtypes
3. HuggingFace models support `attn_implementation="flash_attention_2"` as a one-line config
4. The benefit is in prefill; decode uses specialized kernels (FlashDecoding, PagedAttention)
5. Memory savings from eliminating the N² intermediate allow higher batch sizes or longer contexts